# Hücre 1 — Bağımlılıklar

In [ ]:
!pip install -qU llama-index
!pip install -qU llama-index-embeddings-huggingface
!pip install -qU llama-index-retrievers-bm25
!pip install -qU llama-index-postprocessor-flag-embedding-reranker
!pip install -qU sentence-transformers rank_bm25 nest_asyncio

# Hücre 2 — Import, GPU Tespiti, Yardımcı Fonksiyonlar

In [ ]:
import os
# os.environ["HF_TOKEN"] = ""
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import asyncio
import gc
import torch
import json
import glob
from typing import List

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.core.node_parser import SentenceSplitter, MarkdownNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.postprocessor.flag_embedding_reranker import FlagEmbeddingReranker
from llama_index.core.evaluation import QueryResponseDataset
from llama_index.core import SimpleDirectoryReader
from llama_index.core.indices.query.schema import QueryBundle

# ── GPU TESPİTİ ──
print(f"🖥️  Kullanılabilir GPU: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"   cuda:{i} → {name} ({mem:.1f} GB)")

print("\n⚠️ DİKKAT: Bu notebook 100% stabilite için sadece cuda:0'ı kullanacaktır.")

# ── GLOBAL DEĞİŞKENLER ──
all_results = []

def add_result(group, model_name, mrr, hit_rate):
    all_results.append({
        "Group": group,
        "System": model_name,
        "MRR@5": round(mrr, 4),
        "HitRate@5": round(hit_rate, 4)
    })
    print(f"  ✅ [{group}] {model_name} → MRR: {mrr:.4f} | Hit Rate: {hit_rate:.4f}")

# ── BATCH DEĞERLENDİRİCİ (CancelledError ve OOM önleyici) ──
async def evaluate_retriever_custom(retriever, queries_dict, relevant_docs_dict,
                                     top_k=5, max_concurrent=15, batch_size=100):
    sem = asyncio.Semaphore(max_concurrent)

    async def process_single_query(query_str, expected_ids):
        if not expected_ids:
            return None
        expected_ids_clean = [str(eid).strip().split(".")[0] for eid in expected_ids]

        async with sem:
            try:
                retrieved_nodes = await retriever.aretrieve(query_str)
                retrieved_nodes = retrieved_nodes[:top_k]

                retrieved_doc_ids = []
                for n in retrieved_nodes:
                    possible_ids = []
                    if getattr(n.node, "ref_doc_id", None):
                        possible_ids.append(n.node.ref_doc_id)
                    if getattr(n.node, "node_id", None):
                        possible_ids.append(n.node.node_id)
                    if n.node.metadata and "file_name" in n.node.metadata:
                        possible_ids.append(n.node.metadata["file_name"])

                    for pid in possible_ids:
                        clean_pid = str(pid).strip().split(".")[0]
                        if clean_pid not in retrieved_doc_ids:
                            retrieved_doc_ids.append(clean_pid)

                hit = 1.0 if any(eid in retrieved_doc_ids for eid in expected_ids_clean) else 0.0

                mrr = 0.0
                for rank, doc_id in enumerate(retrieved_doc_ids):
                    if doc_id in expected_ids_clean:
                        mrr = 1.0 / (rank + 1)
                        break
                return (mrr, hit)
            except Exception as e:
                return None

    all_items = list(queries_dict.items())
    all_query_results = []
    total_batches = (len(all_items) + batch_size - 1) // batch_size

    for i in range(0, len(all_items), batch_size):
        batch = all_items[i : i + batch_size]
        batch_num = i // batch_size + 1
        print(f"    Batch {batch_num}/{total_batches} ({len(batch)} sorgu)...", end="\r")

        tasks = [
            process_single_query(q_str, relevant_docs_dict.get(q_id, []))
            for q_id, q_str in batch
        ]
        batch_results = await asyncio.gather(*tasks)
        all_query_results.extend(batch_results)

    print(f"    ✓ {len(all_items)} sorgu tamamlandı.                        ")

    valid_results = [r for r in all_query_results if r is not None]
    if not valid_results:
        return 0.0, 0.0

    avg_mrr = sum(r[0] for r in valid_results) / len(valid_results)
    avg_hit = sum(r[1] for r in valid_results) / len(valid_results)
    return avg_mrr, avg_hit

print("✅ Setup tamamlandı.")

# Hücre 3 — Veri Yükleme

In [ ]:
# BASE_PATH Kendi yolunuza göre ayarlayın
BASE_PATH = "/kaggle/input/datasets/yekbun/turkish-rag-dataset/rag-dataset"

QA_PATH   = os.path.join(BASE_PATH, "benchmark")
DOCS_PATH = os.path.join(BASE_PATH, "stage2_cleaned")

print("1. Dokümanlar yükleniyor...")
reader = SimpleDirectoryReader(input_dir=DOCS_PATH, recursive=True)
documents = reader.load_data()

for doc in documents:
    doc.id_ = doc.metadata["file_name"].replace(".md", "").replace(".txt", "").strip()

print(f"   Toplam Doküman: {len(documents)}")

print("\n2. QA Çiftleri yükleniyor...")
queries = {}
relevant_docs = {}

qa_files = glob.glob(os.path.join(QA_PATH, "*.json"))

for file_path in qa_files:
    try:
        temp_dataset = QueryResponseDataset.from_json(file_path)
        queries.update(temp_dataset.queries)
        for k, v in temp_dataset.relevant_docs.items():
            relevant_docs[k] = relevant_docs.get(k, []) + v
    except Exception:
        with open(file_path, "r", encoding="utf-8") as f:
            qa_data = json.load(f)

        for idx, item in enumerate(qa_data):
            q_id = item.get("id", item.get("query_id", f"{os.path.basename(file_path)}_{idx}"))
            queries[q_id] = item.get("query", item.get("question", ""))

            exp_docs = item.get("source_file", item.get("expected_doc_id", item.get("doc_id", [])))
            if not isinstance(exp_docs, list):
                exp_docs = [exp_docs]
            relevant_docs[q_id] = exp_docs

TEST_LIMIT = None  # Hızlı test için sayı verebilirsiniz (örn: 50)
if TEST_LIMIT and TEST_LIMIT < len(queries):
    queries = dict(list(queries.items())[:TEST_LIMIT])
    relevant_docs = {k: relevant_docs.get(k, []) for k in queries.keys()}

print(f"   Toplam Soru: {len(queries)}")

# Hücre 4 — Group 1: Embedding Modeli Karşılaştırması

In [ ]:
print("\n" + "=" * 60)
print("  GRUP 1: EMBEDDING MODELİ KARŞILAŞTIRMASI")
print("=" * 60)

embedding_models = {
    "RAG-1": "BAAI/bge-m3",
    "RAG-2": "intfloat/multilingual-e5-large",
    "RAG-3": "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
}

group1_results = []

async def _eval_embedding(rag_id, model_name):
    print(f"\n🚀 {rag_id} ({model_name.split('/')[-1]}) değerlendiriliyor...")

    embed_model = HuggingFaceEmbedding(
        model_name=model_name,
        device="cuda:0",
        embed_batch_size=32, # OOM riskini sıfırlamak için 32 yapıldı
    )

    index = VectorStoreIndex.from_documents(
        documents,
        embed_model=embed_model,
        transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=128)],
    )
    retriever = index.as_retriever(similarity_top_k=5)

    mrr, hit_rate = await evaluate_retriever_custom(
        retriever, queries, relevant_docs,
        top_k=5, max_concurrent=20, batch_size=100,
    )

    add_result("Group 1", f"{rag_id} ({model_name.split('/')[-1]})", mrr, hit_rate)
    group1_results.append({"name": model_name, "mrr": mrr})

    del index, retriever, embed_model
    gc.collect()
    torch.cuda.empty_cache()

for rag_id, model_name in embedding_models.items():
    await _eval_embedding(rag_id, model_name)

best_embedding_model = max(group1_results, key=lambda x: x["mrr"])["name"]
print(f"\n🏆 GRUP 1 KAZANANI: {best_embedding_model}")

Settings.embed_model = HuggingFaceEmbedding(
    model_name=best_embedding_model,
    device="cuda:0",
    embed_batch_size=32,
)

# Hücre 5 — Group 2: Chunk Stratejisi Karşılaştırması

In [ ]:
print("\n" + "=" * 60)
print("  GRUP 2: CHUNK STRATEJİSİ KARŞILAŞTIRMASI")
print("=" * 60)

chunk_strategies = {
    "RAG-4 (256/50)":     SentenceSplitter(chunk_size=256, chunk_overlap=50),
    "RAG-5 (1024/200)":   SentenceSplitter(chunk_size=1024, chunk_overlap=200),
    "RAG-6 (Header Based)": MarkdownNodeParser(),
}

group2_results = []

async def _eval_chunk(strategy_name, splitter):
    print(f"\n🚀 {strategy_name} değerlendiriliyor...")

    embed_model = HuggingFaceEmbedding(
        model_name=best_embedding_model,
        device="cuda:0",
        embed_batch_size=32,
    )

    index = VectorStoreIndex.from_documents(
        documents,
        embed_model=embed_model,
        transformations=[splitter],
    )
    retriever = index.as_retriever(similarity_top_k=5)

    mrr, hit_rate = await evaluate_retriever_custom(
        retriever, queries, relevant_docs,
        top_k=5, max_concurrent=20, batch_size=100,
    )

    add_result("Group 2", strategy_name, mrr, hit_rate)
    group2_results.append({"name": strategy_name, "splitter": splitter, "mrr": mrr})

    del index, retriever, embed_model
    gc.collect()
    torch.cuda.empty_cache()

for name, splitter in chunk_strategies.items():
    await _eval_chunk(name, splitter)

best_chunk = max(group2_results, key=lambda x: x["mrr"])
print(f"\n🏆 GRUP 2 KAZANANI: {best_chunk['name']}")

Settings.text_splitter = best_chunk["splitter"]
Settings.embed_model = HuggingFaceEmbedding(
    model_name=best_embedding_model,
    device="cuda:0",
    embed_batch_size=32,
)
best_index = VectorStoreIndex.from_documents(documents)

# Hücre 6 — Group 3: Retrieval Stratejisi (BM25 + Hybrid)

In [ ]:
print("\n" + "=" * 60)
print("  GRUP 3: RETRIEVAL STRATEJİSİ KARŞILAŞTIRMASI")
print("=" * 60)
gc.collect()
torch.cuda.empty_cache()

vector_retriever = best_index.as_retriever(similarity_top_k=5)
nodes = list(best_index.docstore.docs.values())
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5)

hybrid_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=5,
    num_queries=1,
    use_async=True,
)

print("\n🚀 RAG-7 (BM25 Only) değerlendiriliyor...")
mrr_7, hit_7 = await evaluate_retriever_custom(
    bm25_retriever, queries, relevant_docs,
    top_k=5, max_concurrent=20, batch_size=100,
)
add_result("Group 3", "RAG-7 (BM25 Only)", mrr_7, hit_7)

print("\n🚀 RAG-8 (Hybrid: Vector + BM25) değerlendiriliyor...")
mrr_8, hit_8 = await evaluate_retriever_custom(
    hybrid_retriever, queries, relevant_docs,
    top_k=5, max_concurrent=20, batch_size=100,
)
add_result("Group 3", "RAG-8 (Hybrid: Vector + BM25)", mrr_8, hit_8)

# Hücre 7 — Group 4: Reranker Etkisi

In [ ]:
print("\n" + "=" * 60)
print("  GRUP 4: RERANKER ETKİSİ")
print("=" * 60)
gc.collect()
torch.cuda.empty_cache()

reranker = FlagEmbeddingReranker(
    top_n=5,
    model="BAAI/bge-reranker-v2-m3",
    device="cuda:0",
)

class CustomRerankRetriever:
    def __init__(self, base_retriever, reranker_model):
        self.base_retriever = base_retriever
        self.reranker_model = reranker_model

    async def aretrieve(self, query_str):
        self.base_retriever.similarity_top_k = 15
        nodes = await self.base_retriever.aretrieve(query_str)
        return self.reranker_model.postprocess_nodes(
            nodes, query_bundle=QueryBundle(query_str)
        )

rerank_retriever = CustomRerankRetriever(hybrid_retriever, reranker)

print("\n🚀 RAG-9 (Hybrid + Reranker) değerlendiriliyor...")
# Reranker GPU'yu yoğun kullandığı için max_concurrent'i 4'te tutuyoruz
mrr, hit_rate = await evaluate_retriever_custom(
    rerank_retriever, queries, relevant_docs,
    top_k=5, max_concurrent=4, batch_size=50,
)
add_result("Group 4", "RAG-9 (Hybrid + Reranker)", mrr, hit_rate)

# Hücre 8 — Sonuç Tablosu

In [ ]:
print("\n" + "=" * 60)
print("  🏆 TÜM RAG SİSTEMLERİ PERFORMANS TABLOSU 🏆")
print("=" * 60)

df_results = pd.DataFrame(all_results)
display(df_results.sort_values(by=["Group", "MRR@5"], ascending=[True, False]))

df_results.to_csv("rag_evaluation_results.csv", index=False)
print("\n✅ Sonuçlar 'rag_evaluation_results.csv' olarak kaydedildi.")